In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display
from utils.constants import SEC_TO_INEV, KPC_TO_INEV, GCM3_TO_EV4
from utils.data_utils import read_medium_data, interpolate_data
from inputs.spectrum_sources import (CSVSpectrum, AnalyticSpectrum, SpectrumSource,
                                      TimeVaryingSpectrum, plot_spectrum)
from inputs.configs import PhysicsConfig, PropagationConfig
from utils.logging_utils import setup_logging, get_logger
from waveform.collection import WaveformCollection
from waveform.propagation import (plot_spectrogram, export_source_parameters,
                                   create_source_from_propagation, load_source_from_file)

# Enable inline plotting
%matplotlib inline

# Configure logging
setup_logging(log_file='propagation.log', level='INFO')
logger = get_logger()

# Complete Workflow: Spectrum → Propagation → Constraints

This notebook demonstrates the **complete pipeline** from input spectrum to constraint plots.

## What you'll get:
1. **Spectrogram visualization** (PDF) - Shows frequency vs. time evolution
2. **Constraint plot** (PNG) - Shows parameter space constraints
3. **Exported parameters** (optional) - Save for later use

## Workflow:
1. Load spectrum from CSV (bosenova collapse)
2. Propagate through galactic density profile
3. Generate spectrogram
4. Bridge to constraint plotting
5. Generate constraint plot

---

**Note:** For quick constraint plots without propagation, see `examples_constraint_plots_manual.ipynb`

In [ ]:
mass = 1e-19
burst_duration = 400 / (mass * SEC_TO_INEV)

# Load bosenova spectrum from CSV with scaling applied directly
spectrum = CSVSpectrum(
    'Spectra/BosonStarSpectrumRelOnly.txt',
    i_p=0, 
    i_A=1, 
    skip_header=False, 
    num_points=1000,
    scaled_momentum=mass,  # Scale dimensionless momentum by mass
    scaled_amplitude=lambda A: A * (1/1e-85)  # Normalization scaling
)

# Configure physics and propagation
physics = PhysicsConfig(mass=mass, coupling=1e22, K=1e-3, burst_duration=burst_duration)
propagation_config = PropagationConfig('Galactic_Density_Profile.csv', density_num_points=1000)

# For bosenova, scale the density profile distance from 10kpc to 1pc (x/10000)
collection = WaveformCollection(spectrum, physics, propagation_config)

# Override the density profile to apply bosenova scaling
x, rho = read_medium_data('Galactic_Density_Profile.csv', i_R=0, i_rho=2)
x_interp, rho_interp = interpolate_data(x/10000, rho, 1000)  # Bosenova: Convert 10 kpc to 1 pc
collection.density_profile = [x_interp * KPC_TO_INEV, rho_interp * GCM3_TO_EV4]

# Propagate
N_points_spectrogram = 1000
results = collection.propagate_all(N_points_spectrogram=N_points_spectrogram, save_waveform=False)

# Plot spectrogram
avg_density, arrival_window = plot_spectrogram(
    N_points_spectrogram, results['t_min'], results['t_max'],
    results['E'], results['spectrogram'], 
    filename='spectrogram_bosenova.pdf'
)

# Get distance from density profile
R = collection.density_profile[0][-1] - collection.density_profile[0][0]

# Export source parameters with coupling information
export_source_parameters(
    avg_density, burst_duration, R, mass,
    arrival_window=arrival_window,
    coupling=physics.coupling,
    K=physics.K,
    coupling_type='electron',
    coupling_order='quad',
    to_file=True,
    filename='bosenova.param'
)

print("\nExported parameters to bosenova.param")

In [ ]:
from inputs.source import Source
from inputs.spectrum import SignalModel
from inputs.experiment import Experiment
from utils.constants import YEAR_TO_SEC, DAY_TO_SEC
from outputs.output_handler import OutputHandler
from plotting.plots import Plot
from utils.constants import SOLAR_TO_EV

# Create Source from propagation results (direct bridge - no file needed)
source = create_source_from_propagation(
    avg_density, 
    burst_duration, 
    R, 
    mass,
    arrival_window,
    coupling_type='electron',
    coupling_order='quad',
    ULB_type='scalar'
)

print('Created Source from propagation results:')
print(f'  mass = {source.mass} eV')
print(f'  R = {source.R} pc')
print(f'  tstar = {source.tstar} s')
print(f'  Etot = {source.Etot / SOLAR_TO_EV:.4f} solar masses')
print(f'  coupling_type = {source.coupling_type}')
print(f'  coupling_order = {source.coupling_order}')

In [ ]:
# Configure experiment
experiment = Experiment(
    integration_time=DAY_TO_SEC,
    integration_time_DM=YEAR_TO_SEC,
    sensitivity=1e-17,
    time_delays={'day': DAY_TO_SEC, 'year': YEAR_TO_SEC}
)

# Create signal model
signal_model = SignalModel(source=source, experiment=experiment)

print('\nSignal model created - ready for constraint plotting')

In [ ]:
# Generate constraint plot
plot = Plot((1e-21, 6e-6), (5e-21, 1e20), exclude_mass=False)
output = OutputHandler(plot)
output.plot_parameter_space(source, signal_model, plot, save_path='constraints_bosenova.png')

print('Constraint plot saved to constraints_bosenova.png')

print('\n=== Workflow Complete ===')
print('Generated outputs:')
print('  1. spectrogram_bosenova.pdf (from propagation)')
print('  2. constraints_bosenova.png (constraint plot)')
print('  3. bosenova.param (exported parameters - optional for later use)')